# 03 · Join Sofascore + Capology — Turkey Süper Lig 21/22

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2021/22 de Süper Lig turca**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_turkey_2122.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_turkey_2122.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  635 jugadores | 116 columnas
Capology:   692 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   altay
   basaksehir fk
   besiktas jk
   fatih karagumruk
   gaziantep fk
   kasmpasa

En Capology pero no en Sofascore:
   altay sk
   basaksehir
   besiktas
   gaziantep bb
   karagumrukspor
   kasimpasa


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'altay sk':'altay',
            'basaksehir':'basaksehir fk',
            'besiktas':'besiktas jk',
            'gaziantep bb':'gaziantep fk',
            'karagumrukspor':'fatih karagumruk',
            'kasimpasa':'kasmpasa'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 475/635 (74.8%)
Sin emparejar: 160


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          55
Revisión media    (0.75 ≤ score < 0.90):   16
Revisión estricta (0.50 ≤ score < 0.75):   59
Revisión muy est. (score < 0.50):           30


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
79,Dimitris Kolovetsios,Kayserispor,dimitrios kolovetsios,0.976
6,Abdülkerim Bardakcı,Konyaspor,abdulkerim bardakci,0.973
78,Kubilay Kanatsızkuş,Çaykur Rizespor,kubilay kanatsizkus,0.973
116,Mert Miraç Altıntaş,Yeni Malatyaspor,mert mirac altintas,0.973
58,Ertuğrul Taşkıran,Kasımpaşa,ertugrul taskiran,0.970
62,Nicolai Jørgensen,Kasımpaşa,nicolai jorgensen,0.970
83,Şener Özbayraklı,Başakşehir FK,sener ozbayrakli,0.968
13,Dimitris Goutas,Sivasspor,dimitrios goutas,0.968
74,Taylan Antalyalı,Galatasaray,taylan antalyali,0.968
51,Çağtay Kurukalıp,Fenerbahçe,cagtay kurukalip,0.968


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
37,Muammer Yıldırım,Sivasspor,muammer yildirim,0.897
54,Bertuğ Yıldırım,Hatayspor,bertug yildirim,0.889
99,Barış Yardımcı,Konyaspor,baris yardimci,0.880
19,Serkan Kırıntılı,Alanyaspor,serkan kirintili,0.857
44,Serginho,Giresunspor,sergio,0.857
133,Işık Kaan Arslan,Galatasaray,kaan arslan,0.846
126,Osman Ertuğrul Çetin,Fenerbahçe,ertugrul cetin,0.824
77,Houssameddine Ghacha,Antalyaspor,houssam ghacha,0.824
144,Ahmet Eyüp Türkaslan,Yeni Malatyaspor,eyup turkaslan,0.824
22,Magomed Shapi Suleymanov,Giresunspor,shapi suleymanov,0.800


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 16 | Excluidos: 0


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
56,Eren Elmalı,Kasımpaşa,evren eren elmali,0.741
64,Daouda Karamoko Bamba,Altay,daouda bamba,0.727
23,Serge Aka,Altay,serge arnaud aka,0.720
55,Jefferson Junior,Gaziantep FK,jefferson,0.720
39,Campanharo,Kayserispor,gustavo campanharo,0.714
102,Barış Alper Yılmaz,Galatasaray,baris yilmaz,0.714
92,Mickael Tirpan,Kasımpaşa,michal travnik,0.714
85,Joseph William Champness,Giresunspor,joe champness,0.703
152,Kaan Onaran,Sivasspor,hakan arslan,0.696
158,Kaan Yilmaz,Giresunspor,fatih yilmaz,0.696


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['eren elmal',
                    'daouda karamoko bamba',
                    'serge aka',
                    'jefferson junior',
                    'campanharo',
                    'bars alper ylmaz',
                    'joseph william champness',
                    'mahmoud trezeguet',
                    'mehmet feyzi yldrm',
                    'philip awuku gameli',
                    'bahattin demircan',
                    'mahmoud kahraba',
                    'amilton da silva',
                    'carlos ponck',
                    'guilherme haubert sitya'
                    
                    

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 15


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
155,Soner Özdemir,Çaykur Rizespor,gedson fernandes,0.483
149,İzzet Furkan Malak,Göztepe,berkan emir,0.483
143,Emir Dilaver,Çaykur Rizespor,cemali sertel,0.480
67,Engin Aksoy,Hatayspor,eren fansa,0.476
87,Semih Kaya,Galatasaray,omer bayram,0.476
70,Cengizhan Sen,Göztepe,beykan simsek,0.462
49,Hijran Ali Boyaci,Adana Demirspor,benjamin stambouli,0.457
107,Tunay Torun,Kasımpaşa,harun tekin,0.455
69,Ethem Balcı,Kayserispor,emrah bassan,0.455
114,Baris Gun,Giresunspor,doganay aygun,0.455


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 561/635 (88.3%)
Sin salario:     74


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 74


,player,team,minutesPlayed,appearances,goals,assists
0,Damjan Đoković,Adana Demirspor,1750,30,4,0
1,Loïc Rémy,Adana Demirspor,295,10,0,0
2,Hijran Ali Boyaci,Adana Demirspor,18,1,0,0
3,Pa Dibba,Adana Demirspor,14,1,0,0
4,Çağan Kayra Erciyas,Alanyaspor,54,3,0,0
5,Efe Sarıkaya,Altay,91,2,0,0
6,Erdem Ozcan,Altay,79,3,0,0
7,Onur Efe,Altay,9,1,0,0
8,Tugay Gündem,Altay,8,1,0,0
9,Sinan Gümüş,Antalyaspor,118,6,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Adana Demirspor  —  SF sin salario:


,player,minutesPlayed
0,Damjan Đoković,1750
1,Hijran Ali Boyaci,18
2,Loïc Rémy,295
3,Pa Dibba,14


  CG plantilla completa:


,player,player_norm
0,Alper Uludag,alper uludag
1,Arijanet Muric,arijanet muric
2,Benjamin Stambouli,benjamin stambouli
3,Birkir Bjarnason,birkir bjarnason
4,Britt Assombalonga,britt assombalonga
5,David Akintola,david akintola
6,Emrecan Uzun,emrecan uzun
7,Erhun Öztümer,erhun oztumer
8,Erkam Develi,erkam develi
9,Ferhat Kaplan,ferhat kaplan



  Alanyaspor  —  SF sin salario:


...

,player,player_norm
0,Abdulsamed Damlu,abdulsamed damlu
1,Adem Büyük,adem buyuk
2,Aly Malle,aly malle
3,Atakan Müjde,atakan mujde
4,Benjamin Tetteh,benjamin tetteh
5,Berk Yildiz,berk yildiz
6,Bugra Cagiran,bugra cagiran
7,Didier Ndong,didier ndong
8,Eric Ndizeye,eric ndizeye
9,Ertac Özbir,ertac ozbir



  Çaykur Rizespor  —  SF sin salario:


,player,minutesPlayed
0,Efe Tecimer,33
1,Emir Dilaver,24
2,Mahsun Çapkan,8
3,Papiss Demba Cissé,647
4,Soner Özdemir,2


  CG plantilla completa:


,player,player_norm
0,Alberk Koc,alberk koc
1,Alper Potuk,alper potuk
2,Aminu Umar,aminu umar
3,Anil Yasar,anil yasar
4,Aziz Aksoy,aziz aksoy
5,Bryan Dabo,bryan dabo
6,Can Ceylan,can ceylan
7,Cemali Sertel,cemali sertel
8,Damjan Djokovic,damjan djokovic
9,Deniz Hümmet,deniz hummet


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('souza', 'besiktas jk')                    : ('josef', 'besiktas jk'),
    ('munir el kajoui', 'hatayspor')            : ('munir', 'hatayspor'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 2


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: souza (besiktas jk) → josef (besiktas jk)
✅ Match manual aplicado: munir el kajoui (hatayspor) → munir (hatayspor)

Tras matches manuales: 563/635 (88.7%)


In [21]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [22]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_turkey_2122.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_turkey_2122.csv
   Jugadores totales:  635
   Con salario:        563
   Sin salario (NaN):  72
   Columnas:           121
